# RSNA Knee — Training

Thin shell: pulls `knee` from GitHub at a pinned commit, trains on the pre-mounted
competition data, exports checkpoints to `/kaggle/working`.

Current experiment: **E003-dinov2-frozen** — same three per-plane fluid specialists
as E001 (frozen backbone + linear head, gold studies) but with a frozen DINOv2
ViT-S/14 backbone at its native 518px instead of ImageNet ResNet-34 at 224px. The
CV cell runs a same-seed `gold58-cv` A/B against the ResNet baseline. Requires the
`WANDB_API_KEY` Kaggle secret and internet (DINOv2 weights download at train time).

In [ ]:
# Pin a commit so every checkpoint traces to exact code. Training notebooks have internet.
# --no-deps everywhere: Kaggle's image already ships torch/timm/sklearn/numpy compiled
# together; letting pip resolve our pins upgrades numpy and breaks the whole stack.
# STALE for E003 — bump to the dinov2-backbone squash merge before `kaggle kernels push`
# (this run needs knee.cv_gold and DINOV2_BACKBONE, which don't exist at 4ef6afc).
COMMIT = "4ef6afc"
%pip install -q --no-deps "git+https://github.com/Josie29/capstone-rsna-knee@{COMMIT}#egg=knee"
%pip install -q --no-deps pydicom pylibjpeg pylibjpeg-libjpeg pylibjpeg-openjpeg pylibjpeg-rle

import numpy  # fail fast if the image stack is broken or missing
import timm
import torch

print("numpy", numpy.__version__, "| torch", torch.__version__, "| timm", timm.__version__)

from knee.series import SeriesType
from knee.train_gold import train_gold

In [ ]:
# Gold-58 prototype trains straight off the mounted competition data; the private
# mined-labels dataset joins here later (issue #2).
from pathlib import Path

SLUG = "rsna-knee-abnormality-detection"
# Kaggle mounts competitions under /kaggle/input/competitions/<slug> (newer layout)
# or /kaggle/input/<slug> (older docs/examples); accept either.
candidates = [Path("/kaggle/input/competitions") / SLUG, Path("/kaggle/input") / SLUG]
COMP_ROOT = next((p for p in candidates if (p / "train.csv").exists()), None)
if COMP_ROOT is None:
    listing = {str(d): [c.name for c in d.iterdir()] for d in Path("/kaggle/input").iterdir() if d.is_dir()}
    raise FileNotFoundError(f"competition data not found; mounts: {listing}")
print("competition root:", COMP_ROOT)

In [ ]:
# Metrics/hyperparams only -- no report text, no StudyInstanceUIDs (rule 2.4.b).
# Best-effort: the run proceeds without wandb if the secret isn't configured.
run = None
try:
    import wandb
    from kaggle_secrets import UserSecretsClient

    wandb.login(key=UserSecretsClient().get_secret("WANDB_API_KEY"))
    run = wandb.init(project="rsna-knee", config={"commit": COMMIT})
except Exception as exc:  # noqa: BLE001 — telemetry must never kill a training run
    print(f"wandb disabled: {exc}")

In [ ]:
# E003: one specialist per fluid plane (strict typing — no fallback series; studies
# lacking a plane are skipped by that model). Backbone swapped to frozen DINOv2
# ViT-S/14: self-supervised features built for exactly this linear-probe regime.
# 518px is the ViT's fixed native input (patch 14 x 37) — do not change one without
# the other. Baseline config kept below for the CV A/B.
from knee.model import DEFAULT_BACKBONE, DINOV2_BACKBONE

SERIES_TYPES = [SeriesType.SAGITTAL_FLUID, SeriesType.CORONAL_FLUID, SeriesType.AXIAL_FLUID]
BACKBONE = DINOV2_BACKBONE
INPUT_SIZE = 518
BASELINE_BACKBONE = DEFAULT_BACKBONE  # resnet34, E001/E002 regime
BASELINE_INPUT_SIZE = 224

# Trains on the gold studies, must never be evaluated in-sample against them.
def checkpoint_path(series_type: SeriesType) -> Path:
    return Path(f"/kaggle/working/gold58_dinov2_{series_type.value}.pt")

In [ ]:
# Competition DICOMs are pre-mounted read-only; ~58 series per model. 518px ViT
# forwards want the GPU — pick T4/L4 in the UI (machine_shape is set, verify anyway).
results = []
for series_type in SERIES_TYPES:
    result = train_gold(
        COMP_ROOT,
        checkpoint_path(series_type),
        series_type=series_type,
        backbone=BACKBONE,
        input_size=INPUT_SIZE,
    )
    results.append(result)
    print(f"{series_type.value}: trained on {result.n_studies}, skipped {len(result.skipped)}")
    # In-sample only (trains on all gold rows): proves the features carry signal, nothing more.
    print(result.in_sample_auc)

In [ ]:
# Local eval (DECISIONS.md #4): pooled out-of-fold stratified CV of the full ensemble
# on the gold studies, run as a same-seed A/B — new backbone vs the E001/E002 ResNet
# baseline. Identical folds and combiner, so the delta is attributable to the
# backbone. Features extract once per backbone (frozen), so this adds only minutes.
from knee.cv_gold import collect_gold_features, cross_validate_gold

cv_by_backbone = {}
for name, size in [(BACKBONE, INPUT_SIZE), (BASELINE_BACKBONE, BASELINE_INPUT_SIZE)]:
    bank = collect_gold_features(COMP_ROOT, series_types=SERIES_TYPES, backbone=name, input_size=size)
    cv_by_backbone[name] = cross_validate_gold(bank)

for name, cv in cv_by_backbone.items():
    print(f"{name}: macro OOF AUC {cv.macro_auc:.3f} over {cv.n_repeats} repeats: "
          + ", ".join(f"{m:.3f}" for m in cv.macro_auc_per_repeat))
    print({label: round(auc, 3) for label, auc in cv.per_label_auc.items()})

In [ ]:
# Checkpoints are already in /kaggle/working, which persists as notebook output;
# publish all three as the knee-weights dataset so the inference notebook can attach
# them — REPLACING the old ResNet .pt files (duplicate series types make inference raise).
import math

if run is not None:
    run.config.update({"backbone": BACKBONE, "input_size": INPUT_SIZE})
    for result in results:
        wandb.log(
            {
                f"in_sample_auc/{result.series_type.value}/{label}": auc
                for label, auc in result.in_sample_auc.items()
                if not math.isnan(auc)
            }
        )
    for name, cv in cv_by_backbone.items():
        wandb.log(
            {
                f"cv/macro_auc/{name}": cv.macro_auc,
                **{f"cv/auc/{name}/{label}": auc for label, auc in cv.per_label_auc.items() if not math.isnan(auc)},
            }
        )
    run.finish()